In [ ]:
!git clone https://github.com/dshipley71/mcp-rag.git

In [ ]:
%cd mcp-rag

In [ ]:
%ls

In [ ]:
!pip install -r requirements.txt

In [ ]:
import pytest
!pytest tests/test_structure.py

In [ ]:
import pytest
!pytest tests/test_config_loader.py

In [ ]:
import pytest
!pytest tests/test_orchestrator.py

In [ ]:
import os
import shutil
from pathlib import Path
import subprocess

REPO_ROOT = Path("/content/mcp-rag").resolve()
DOCS_DIR = REPO_ROOT / "docs"
DB_DIR = (REPO_ROOT / ".rag" / "velocirag").resolve()

os.chdir(REPO_ROOT)

assert DOCS_DIR.exists(), f"Missing docs directory: {DOCS_DIR}"
assert (DOCS_DIR / "VelociRAG.md").exists(), f"Missing test corpus file: {DOCS_DIR / 'VelociRAG.md'}"

# Clean rebuild
if DB_DIR.exists():
    shutil.rmtree(DB_DIR)

DB_DIR.parent.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR:", DOCS_DIR)
print("DB_DIR:", DB_DIR)

result = subprocess.run(
    ["velocirag", "index", str(DOCS_DIR), "--db", str(DB_DIR)],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

assert result.returncode == 0, "VelociRAG indexing failed"
assert DB_DIR.exists(), f"DB directory was not created: {DB_DIR}"

In [ ]:
import asyncio
import nest_asyncio
import rich
from pathlib import Path

from src.mcp_runtime import MCPRuntime

nest_asyncio.apply()

REPO_ROOT = Path("/content/mcp-rag").resolve()
DB_DIR = (REPO_ROOT / ".rag" / "velocirag").resolve()

async def debug_runtime():
    runtime = MCPRuntime(db_dir=str(DB_DIR))
    await runtime.connect()

    try:
        tools = await runtime.velocirag.list_tools()
        print("TOOLS:", tools)

        health = await runtime.velocirag.call_tool("health", {})
        rich.print("HEALTH:", health)

        search = await runtime.velocirag.call_tool(
            "search",
            {"query": "VelociRAG?", "limit": 3}
        )
        rich.print("SEARCH:", search)
    finally:
        await runtime.close()

await debug_runtime()

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from src.mcp_runtime import MCPRuntime
from src.orchestrator import run_query
from src.models import QueryRequest
from pathlib import Path

REPO_ROOT = Path("/content/mcp-rag").resolve()
DB_DIR = (REPO_ROOT / ".rag" / "velocirag").resolve()

runtime = MCPRuntime(db_dir=str(DB_DIR))
await runtime.connect()

result = await run_query(
    QueryRequest(query="What does VelociRAG do?"),
    runtime
)

rich.print(result)

await runtime.close()